In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Load the data and build the search index:

In [6]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

### Defining the tool

First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The **description** is the most important field, because the model reads it to decide when to call the function. **parameters** is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

### The Agentic Loop
A developer prompt:

We make that more reliable with a developer message that spells out how to behave. This is where we give the agent its role. The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [9]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

**A function-call helper:**

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [10]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more if branches (or switch to a registry).

**Processing one response**
Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

In [11]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course just discovered discovered course can I join"}
function_call: search {"query":"enroll join after start late registration add course"}
function_call: search {"query":"course enrollment policy start date waitlist"}
function_call: search {"query":"can students join after enrollment deadline"}


The has_function_calls flag tells us whether the model needs another API call. If the response contains a function call, the updated messages has tool output the model hasn't seen yet. We'll need to send it back.

### The full agent loop
We wrap this in a *while* loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

In [12]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. You can also start learning and working through the materials right away.

If you'd like, I can also help you figure out:
- how to start the course,
- whether you can still get a certificate,
- or where to find the course materials.

Anything else you want to explore?


This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

The **exit condition** is the simplest one possible. No function calls this turn means we're done. Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

**Wrapping it in a function**

Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.

In [14]:
def agent_loop(instructions, question, model="gpt-5.4-nano") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

**Testing with a question with a typo**

In [15]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install"}
function_call: search {"query":"Olama local setup run model"}
function_call: search {"query":"Ollama FAQ download models localhost"}
function_call: search {"query":"course FAQ Ollama docker"}
function_call: search {"query":"Ollama command \"ollama\" run"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, follow these steps:

### 1) Install Ollama
Go to the download page: https://ollama.com/download

- **Linux** (terminal):
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

### 2) Download + run a model
In a terminal:
```bash
ollama run llama3
```
This will download the model (if needed) and start an interactive chat.

### 3) (Optional) Start/check the local Ollama server
Ollama uses a local HTTP server on **http://localhost:11434**.

Test it:
```bash
curl http://localhost:11434
```
You should see something like a JSON response with models.

### 4) Use it from Python (course-friendly)
Install the cl

'To run **Ollama locally**, follow these steps:\n\n### 1) Install Ollama\nGo to the download page: https://ollama.com/download\n\n- **Linux** (terminal):\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### 2) Download + run a model\nIn a terminal:\n```bash\nollama run llama3\n```\nThis will download the model (if needed) and start an interactive chat.\n\n### 3) (Optional) Start/check the local Ollama server\nOllama uses a local HTTP server on **http://localhost:11434**.\n\nTest it:\n```bash\ncurl http://localhost:11434\n```\nYou should see something like a JSON response with models.\n\n### 4) Use it from Python (course-friendly)\nInstall the client:\n```bash\npip install ollama\n```\n\nMinimal example:\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model="llama3",\n    messages=[{"role": "user", "content": "Hello! Summarize RAG in one sentence."}]\n)\n\nprint(response["message"]["content"])\n```\n\n### If you get “connection refused”\nRestart the server (comm

**Also try the course enrollment question:**

In [16]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course discovered course late can I still join"}
function_call: search {"query":"enrollment deadline still join"}
function_call: search {"query":"late enrollment policy course"}
function_call: search {"query":"start date can join after start"}
function_call: search {"query":"add to course after registration"}
iteration #2...
ASSISTANT:
Yes—you can still join. For the **certificate**, the key requirement is that you **submit your Capstone project while submissions are still open**.

If you want to start right away, you can begin with the course materials anytime (videos + GitHub notebooks), and then follow the weekly deadlines on the course platform.

Are you joining mainly for the **certificate**, or just to learn and do the materials?


'Yes—you can still join. For the **certificate**, the key requirement is that you **submit your Capstone project while submissions are still open**.\n\nIf you want to start right away, you can begin with the course materials anytime (videos + GitHub notebooks), and then follow the weekly deadlines on the course platform.\n\nAre you joining mainly for the **certificate**, or just to learn and do the materials?'

**Encouraging multiple searches**

There's a subtle issue here. The model often answers after the first search, even when more searches would help. It reasons that it already knows enough, so why bother. We push it to explore more by rewriting the instructions.

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered just discovered can I join course"}
iteration #2...
ASSISTANT:
Yes—you can still join, even if you just discovered the course. If you want to receive a certificate, make sure you submit your project while submissions are still open.

Do you mean joining for **certificate** or just to **follow the lessons/homework**?


'Yes—you can still join, even if you just discovered the course. If you want to receive a certificate, make sure you submit your project while submissions are still open.\n\nDo you mean joining for **certificate** or just to **follow the lessons/homework**?'

### Restricting off-topic questions
Right now the agent will answer anything. Ask it about chess and it will still try.

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ. For our own use we might be fine letting it answer from general knowledge. So treat this mainly as an illustration of steering scope.


In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit course FAQ"}
iteration #2...
ASSISTANT:
It sounds like you’re asking about the chess term **“Queen’s Gambit”** (often misspelled as “queen gambit”). However, that’s **not something covered in the course FAQ** I have access to—I only see course logistics questions (deadlines, certificates, enrollment, etc.), not chess topics.

If you meant a **course-specific** “Queen(s) Gambit” (e.g., an assignment/module name), tell me which part of the course you saw it in (week/module/homework link), and I can check our materials/FAQ for that.  

Are there other course logistics questions you wanted to explore (enrollment, deadlines, certificates, workflow)?


'It sounds like you’re asking about the chess term **“Queen’s Gambit”** (often misspelled as “queen gambit”). However, that’s **not something covered in the course FAQ** I have access to—I only see course logistics questions (deadlines, certificates, enrollment, etc.), not chess topics.\n\nIf you meant a **course-specific** “Queen(s) Gambit” (e.g., an assignment/module name), tell me which part of the course you saw it in (week/module/homework link), and I can check our materials/FAQ for that.  \n\nAre there other course logistics questions you wanted to explore (enrollment, deadlines, certificates, workflow)?'

This is a lightweight form of an **input guardrail**. We tell the agent what's in scope and what isn't. A real guardrail checks the input before the agent runs and can block off-topic questions outright. That's a separate topic, but instructions are the first place to start.

This handwritten loop is the best way to understand what frameworks hide from you. Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.